# Exploratory Data Analysis: Animals-10 Dataset

This notebook performs a simple EDA on the [Animals-10](https://www.kaggle.com/datasets/alessiocorrado99/animals10) image classification dataset used for training the ResNet-50 animal classifier.

The dataset contains images of 10 animal classes with Italian folder names that we remap to English.

In [ ]:
import os
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from PIL import Image

# Italian-to-English class name mapping
TRANSLATE = {
    "cane":       "dog",
    "cavallo":    "horse",
    "elefante":   "elephant",
    "farfalla":   "butterfly",
    "gallina":    "chicken",
    "gatto":      "cat",
    "mucca":      "cow",
    "pecora":     "sheep",
    "ragno":      "spider",
    "scoiattolo": "squirrel",
}

# Update this path to where your dataset is stored
DATA_DIR = Path("/kaggle/input/animals10/raw-img")

# Collect all image paths and their class labels
image_paths = []
labels = []
for folder in sorted(DATA_DIR.iterdir()):
    if folder.is_dir() and folder.name in TRANSLATE:
        class_name = TRANSLATE[folder.name]
        for img_file in folder.iterdir():
            if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".gif"):
                image_paths.append(img_file)
                labels.append(class_name)

print(f"Total images: {len(image_paths)}")
print(f"Number of classes: {len(set(labels))}")

## 1. Class Distribution

Check how many images each class has and whether the dataset is balanced.

In [ ]:
class_counts = Counter(labels)
classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(classes, counts, color="steelblue", edgecolor="black")
ax.set_xlabel("Animal Class")
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution in Animals-10 Dataset")
ax.bar_label(bars)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Print stats
print(f"\nMin: {min(counts)} ({classes[counts.index(min(counts))]})")
print(f"Max: {max(counts)} ({classes[counts.index(max(counts))]})")
print(f"Mean: {np.mean(counts):.0f}")
print(f"Std: {np.std(counts):.0f}")

## 2. Sample Images

Display a few random images from each class to get a visual sense of the data.

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for idx, class_name in enumerate(classes):
    # Get all images for this class
    class_images = [p for p, l in zip(image_paths, labels) if l == class_name]
    sample_path = np.random.choice(class_images)

    img = Image.open(sample_path).convert("RGB")
    axes[idx].imshow(img)
    axes[idx].set_title(f"{class_name}\n({img.size[0]}x{img.size[1]})")
    axes[idx].axis("off")

plt.suptitle("Random Sample from Each Class", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Image Resolution Distribution

Analyze the range of image sizes across the dataset to understand what preprocessing (resize/crop) needs to handle.

In [ ]:
# Sample a subset for faster processing
sample_size = min(2000, len(image_paths))
sampled_paths = np.random.choice(image_paths, size=sample_size, replace=False)

widths = []
heights = []
aspect_ratios = []

for p in sampled_paths:
    try:
        img = Image.open(p)
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(w / h)
    except Exception:
        continue

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=50, color="steelblue", edgecolor="black")
axes[0].set_title("Image Widths")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")

axes[1].hist(heights, bins=50, color="coral", edgecolor="black")
axes[1].set_title("Image Heights")
axes[1].set_xlabel("Height (px)")

axes[2].hist(aspect_ratios, bins=50, color="seagreen", edgecolor="black")
axes[2].set_title("Aspect Ratios (W/H)")
axes[2].set_xlabel("Ratio")

plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}")